Method - 01

In [ ]:
# ===============================================================
# Smart Research Paper Summarizer - Prototype (Partial Implementation)
# ===============================================================

# Install dependencies
!pip install PyMuPDF transformers torch rouge-score

# ---------------------------------------------------------------
# 1. Imports
# ---------------------------------------------------------------
import fitz  # PyMuPDF
from transformers import pipeline
import re
import pandas as pd
from rouge_score import rouge_scorer

# ---------------------------------------------------------------
# 2. Summarizer Setup
# ---------------------------------------------------------------
summariser = pipeline("summarization", model="facebook/bart-large-cnn")

# ---------------------------------------------------------------
# 3. PDF Text Extraction
# ---------------------------------------------------------------
def extract_text_from_pdf(pdf_path):
    text = ""
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text += page.get_text("text") + "\n"
    return text

# Example: replace with your actual PDF path
pdf_path = "/home/cdac/Office Projects/Research-Paper-Summarizer/data/2004.05150v2.pdf"  # <-- Put your test PDF file here
full_text = extract_text_from_pdf(pdf_path)
print("PDF text extracted, length:", len(full_text))

# ---------------------------------------------------------------
# 4. Section Detection (Simple Heuristics)
# ---------------------------------------------------------------
def split_into_sections(text):
    sections = re.split(r"\n(?=(Introduction|Methodology|Results|Conclusion|Discussion))", text, flags=re.IGNORECASE)
    structured = {}
    current_title = None
    for s in sections:
        s = s.strip()
        if not s:
            continue
        if s.lower() in ["introduction", "methodology", "results", "conclusion", "discussion"]:
            current_title = s.capitalize()
            structured[current_title] = ""
        elif current_title:
            structured[current_title] += s + "\n"
    return structured

sections = split_into_sections(full_text)
print(" Detected sections:", list(sections.keys()))

# ---------------------------------------------------------------
# 5. Section-wise Summarisation
# ---------------------------------------------------------------
def summarize_section(section_text):
    try:
        return summariser(section_text, max_length=120, min_length=40, do_sample=False)[0]['summary_text']
    except Exception:
        return section_text[:300]  # fallback for very long sections

structured_summary = {}
for title, content in sections.items():
    print(f"\n🔹 Summarizing section: {title}")
    structured_summary[title] = summarize_section(content)

# ---------------------------------------------------------------
# 6. Baseline Summary (Full Text)
# ---------------------------------------------------------------
baseline_summary = summariser(full_text[:1000], max_length=150, min_length=50, do_sample=False)[0]['summary_text']

# ---------------------------------------------------------------
# 7. Structured Summary Output
# ---------------------------------------------------------------
summary_doc = f"""
 Structured Summary
=====================
Title: Example Research Paper
Authors: Sample Author et al. (2024)

Research Question / Objective:
{structured_summary.get('Introduction', 'N/A')}

Methodology Summary:
{structured_summary.get('Methodology', 'N/A')}

Key Results / Findings:
{structured_summary.get('Results', 'N/A')}

Contributions:
{structured_summary.get('Discussion', 'N/A')}

Limitations / Future Work:
{structured_summary.get('Conclusion', 'N/A')}
"""

print(summary_doc)

# ---------------------------------------------------------------
# 8. Evaluation (ROUGE Comparison)
# ---------------------------------------------------------------
def compute_rouge(summary_a, summary_b):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = scorer.score(summary_a, summary_b)
    return {k: v.fmeasure for k, v in scores.items()}

rouge_scores = compute_rouge(baseline_summary, summary_doc)
print("\nROUGE comparison (Baseline vs Structured):")
for metric, val in rouge_scores.items():
    print(f"{metric}: {val:.3f}")

# ---------------------------------------------------------------
# 9. Save Output (Optional)
# ---------------------------------------------------------------
with open("structured_summary_output.txt", "w") as f:
    f.write(summary_doc)
print("\nSummary saved to structured_summary_output.txt")

/home/cdac/Office Projects/Research-Paper-Summarizer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use cpu


PDF text extracted, length: 70595
 Detected sections: ['Introduction', 'Results', 'Conclusion']

🔹 Summarizing section: Introduction

🔹 Summarizing section: Results

🔹 Summarizing section: Conclusion

 Structured Summary
Title: Example Research Paper
Authors: Sample Author et al. (2024)

Research Question / Objective:
Introduction
Transformers (Vaswani et al., 2017) have achieved
state-of-the-art results in a wide range of natu-
ral language tasks including generative language
modeling (Dai et al., 2019; Radford et al., 2019)
and discriminative language understanding (De-
vlin et al., 2019). This success is partl

Methodology Summary:
N/A

Key Results / Findings:
Results
Main Result
Tab. 7 summarizes the results of all
our ﬁnetuning experiments. We observe that Long-
former consistently outperforms the RoBERTa
baseline. Its performance gain is especially ob-
vious for tasks that require long context such as
WikiHop and Hyperpartisan. For TriviaQA, the
improv

Contributions:
N/A

Limita

Method - 02 - FINAL!

In [3]:
!pip install PyMuPDF -q
import fitz
import re
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from heapq import nlargest
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize
from transformers import pipeline
import json
import textwrap

[nltk_data] Downloading package punkt to /home/cdac/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/cdac/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [5]:
def extract_text(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []
    for page in doc:
        pages.append(page.get_text("text"))
    full = "\n\n".join(pages)
    return full

def extract_title_authors(text):
    # Use first 4000 chars (first page) to find title & authors
    head = text[:4000]
    # Split into lines and remove empty lines
    lines = [l.strip() for l in head.splitlines() if l.strip()]
    # Heuristics:
    # - Title likely first non-empty line with more than 3 words and not all-caps.
    # - Authors often on the next line(s) with commas/@ or affiliation keywords.
    title = lines[0] if lines else ""
    # try to find the authors line(s): look for lines with '@' or commas and short length
    authors = None
    for i in range(1, min(10, len(lines))):
        ln = lines[i]
        if "@" in ln or re.search(r'\b(University|Institute|Laboratory|Dept|Department|Allen)\b', ln, flags=re.I) or (len(ln.split())<=10 and "," in ln):
            # take previous line(s) combined if those look like names
            candidate = lines[i-1]
            # sometimes authors span two lines; include the previous two if looks right
            if i-2 >=0 and len(lines[i-2].split())<=10 and not re.search(r'Abstract', lines[i-2], flags=re.I):
                candidate = lines[i-2] + " ; " + candidate
            authors = candidate
            break
    if authors is None:
        # fallback: pick second line
        authors = lines[1] if len(lines)>1 else ""
    return title.strip(), authors.strip()

def detect_sections(text):
    lines = text.splitlines()
    headings = []
    for i, ln in enumerate(lines):
        for pattern in HEADING_REGEX:
            if re.match(pattern, ln, flags=re.I):
                headings.append((i, ln.strip()))
                break
    # If we didn't get headings, attempt to find known title words as section markers
    if not headings:
        # fallback: look for common single words
        for i, ln in enumerate(lines):
            if re.match(r'^\s*Introduction\b', ln, flags=re.I):
                headings.append((i, ln.strip()))
    # Build sections by slicing lines
    sections = {}
    for idx, (line_no, header) in enumerate(headings):
        start = line_no + 1
        end = headings[idx+1][0] if idx+1 < len(headings) else len(lines)
        sec_name = re.sub(r'[^A-Za-z0-9 ]','',header).strip()
        content = "\n".join(lines[start:end]).strip()
        sections[sec_name] = content
    # If headings empty, put whole doc into "body"
    if not sections:
        sections = {"body": text}
    return sections

#-------------------------------------------------------------------------------
# Set your PDF path (uploaded file)
PDF_PATH = "/home/cdac/Office Projects/Research-Paper-Summarizer/data/2004.05150v2.pdf"   #your uploaded file
OUT_JSON = "structured_summary.json"
# canonical headings we expect
HEADING_REGEX = [
    r'^\s*Abstract\b', r'^\s*1\s*Introduction\b', r'^\s*Introduction\b',
    r'^\s*Related Work\b', r'^\s*2\s*Related Work\b',
    r'^\s*3\s*Longformer\b', r'^\s*3\s*Longformer\b',
    r'^\s*Implementation\b', r'^\s*Methodology\b', r'^\s*Pretraining\b',
    r'^\s*Pretraining and Finetuning\b', r'^\s*4\s*Autoregressive\b',
    r'^\s*Results\b', r'^\s*4.2 Results\b', r'^\s*Evaluation\b',
    r'^\s*Conclusion\b', r'^\s*8\s*Conclusion\b', r'^\s*Conclusion and Future Work\b',
    r'^\s*References\b'
]

summariser = pipeline("summarization", model="facebook/bart-large-cnn", device=-1)

raw_text = extract_text(PDF_PATH)
print("Extracted text length:", len(raw_text))

title, authors = extract_title_authors(raw_text)
print("Title:", title)
print("Authors:", authors)

sections = detect_sections(raw_text)
print("Detected sections:", list(sections.keys()))



Device set to use cpu


Extracted text length: 70610
Title: Longformer: The Long-Document Transformer
Authors: Matthew E. Peters∗ ; Arman Cohan∗
Detected sections: ['Abstract', 'Introduction', 'Related Work', 'implementation 3 also includes a custom CUDA', 'Implementation', 'Evaluation', 'Results', 'Pretraining and Finetuning', 'Pretraining', 'Conclusion and Future Work', 'References', 'Implementation Details', 'Pretraining Data']


In [6]:
# Cell 5
def extractive_top_sentences(text, top_k=6):
    sents = sent_tokenize(text)
    if len(sents) <= top_k:
        return sents
    vect = TfidfVectorizer(stop_words='english').fit_transform(sents)
    scores = vect.sum(axis=1).A1
    top_idx = nlargest(top_k, range(len(scores)), key=lambda i: scores[i])
    selected = [sents[i] for i in sorted(top_idx)]
    return selected

def hybrid_section_summary(section_text, k=6, do_rewrite=True):
    # 1) extract top-k sentences (extractive)
    selected = extractive_top_sentences(section_text, top_k=k)
    extractive = " ".join(selected)
    if not do_rewrite or len(extractive.split())<20:
        return extractive  # keep as-is
    # 2) rewrite (concise abstractive) — small safe rewrite: summariser on extractive text
    try:
        out = summariser(extractive, max_length=120, min_length=30, do_sample=False)[0]['summary_text']
        return out
    except Exception as e:
        # fallback: return extractive text (no hallucination)
        print("Rewrite failed:", e)
        return extractive

# Build summaries for canonical sections
canonical_map = {
    "Introduction":"Research Question / Objective",
    "Abstract":"Abstract",
    "Pretraining and Finetuning":"Methodology / Pretraining",
    "Pretraining":"Methodology / Pretraining",
    "Results":"Key Results / Findings",
    "Evaluation":"Key Results / Findings",
    "Conclusion and Future Work":"Conclusion / Limitations / Future Work",
    "Conclusion":"Conclusion / Limitations / Future Work"
}

summaries = {}
for sec, content in sections.items():
    # Map header to canonical; fallback to sec name
    key = canonical_map.get(sec, sec)
    print(f"\nProcessing section: {sec} -> {key} (chars: {len(content)})")
    summaries[key] = hybrid_section_summary(content, k=6, do_rewrite=True)

# Also assemble baseline summary from Abstract if present
baseline = summaries.get("Abstract", None)
if baseline is None:
    # fallback: summarize first 4000 chars using sliding window approach (naive)
    snippet = raw_text[:4000]
    baseline = summariser(snippet, max_length=150, min_length=50, do_sample=False)[0]['summary_text']

print("\n--- Baseline (Abstract or whole-text fallback) ---\n", baseline)



Processing section: Abstract -> Abstract (chars: 1178)

Processing section: Introduction -> Research Question / Objective (chars: 5295)

Processing section: Related Work -> Related Work (chars: 827)

Processing section: implementation 3 also includes a custom CUDA -> implementation 3 also includes a custom CUDA (chars: 7937)

Processing section: Implementation -> Implementation (chars: 4599)


Your max_length is set to 120, but your input_length is only 63. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=31)



Processing section: Evaluation -> Key Results / Findings (chars: 236)

Processing section: Results -> Key Results / Findings (chars: 8442)

Processing section: Pretraining and Finetuning -> Methodology / Pretraining (chars: 2469)

Processing section: Pretraining -> Methodology / Pretraining (chars: 6673)

Processing section: Conclusion and Future Work -> Conclusion / Limitations / Future Work (chars: 1275)

Processing section: References -> References (chars: 10715)

Processing section: Implementation Details -> Implementation Details (chars: 4815)

Processing section: Pretraining Data -> Pretraining Data (chars: 13657)
Rewrite failed: index out of range in self

--- Baseline (Abstract or whole-text fallback) ---
 Transformer-based models are unable to pro-cess long sequences due to their self-attention. To address this limitation, we introduce the Longformer with an attentionmechanism that scales linearly with sequence length. Our pretrained Longformer consistently out-performs RoBER

In [7]:
# json format
structured = {
    "metadata": {
        "title": title,
        "authors": authors,
        "source_file": str(Path(PDF_PATH).name)
    },
    "baseline_summary": baseline,
    "sections": summaries
}

# Save JSON for appendix
with open(OUT_JSON, "w", encoding="utf-8") as f:
    json.dump(structured, f, indent=2, ensure_ascii=False)
print(f"\nSaved structured summary to {OUT_JSON}")



Saved structured summary to structured_summary.json


In [8]:
# Cell 6  –  Save report-formatted summary as .txt (not JSON)

OUTPUT_TXT = "Structured_Summary.txt"

def format_for_report(meta, sections):
    # helper to wrap lines neatly
    def wrap(txt, width=95):
        return "\n".join(textwrap.wrap(txt, width=width))

    lines = []
    lines.append("Paper metadata")
    lines.append(f" Title: {meta.get('title','N/A')}")
    lines.append(f" Authors: {meta.get('authors','N/A')}")
    lines.append(" Year: 2020\n")   # extracted from arXiv ID 2004.xxxx

    lines.append("Research Question / Objective")
    lines.append(wrap(sections.get("Research Question / Objective", sections.get("Introduction",""))))
    lines.append("\nMethodology Summary")
    lines.append(wrap(sections.get("Methodology / Pretraining", "")))
    lines.append("\nKey Results / Findings")
    lines.append(wrap(sections.get("Key Results / Findings", "")))
    lines.append("\nContributions")
    # For contribution, merge intro + conclusion
    contrib = sections.get("Conclusion / Limitations / Future Work", "")[:600]
    if not contrib:
        contrib = sections.get("Abstract", "")
    lines.append(wrap(contrib))
    lines.append("\nLimitations / Future Work / Research Gap")
    lines.append(wrap(sections.get("Conclusion / Limitations / Future Work", "")))

    return "\n".join(lines)

# Build metadata
meta = {"title": title, "authors": authors, "year": "2020"}

formatted_summary = format_for_report(meta, summaries)

# --- print sample output for verification ---
print("="*60)
print(formatted_summary[:2500])   # truncated print
print("="*60)

# Save to text file
with open(OUTPUT_TXT, "w", encoding="utf-8") as f:
    f.write(formatted_summary)

print(f"\nStructured report-format summary saved to {OUTPUT_TXT}")


Paper metadata
 Title: Longformer: The Long-Document Transformer
 Authors: Matthew E. Peters∗ ; Arman Cohan∗
 Year: 2020

Research Question / Objective
Longformer is an advan-                tage for natural language tasks such as long docu-     
         ment classiﬁcation, question answering (QA), and coreference resolution. We evaluate
Longformer on autoregressivecharacter-level language modeling using a com-              
 bination of windowed and a new dilated attentionpattern. After pretraining, we apply it to
downstream language tasks through ﬁnetuning and demonstrating that Longformer consistently
outperforms RoBERTa.

Methodology Summary
We evaluate on IMDB (Maas et al., 2011) and Hy-perpartisan news detection. We used three
datasets: WikiHop, TriviaQA, and HotpotQA. Both models are trained for 65K gradient updates
with sequences of 4,096 tokens.

Key Results / Findings
All published top performing models in. this task (Tu et al., 2019; Fang et al. 2020; Shao.et
al., 2020) use

In [9]:
# Cell 7 (optional)
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
res = scorer.score(structured['baseline_summary'], " ".join(structured['sections'].values()))
print("ROUGE (baseline vs combined sections):")
print({k: float(v.fmeasure) for k,v in res.items()})


ROUGE (baseline vs combined sections):
{'rouge1': 0.10744580584354382, 'rouge2': 0.10576015108593013, 'rougeL': 0.10744580584354382}


**Final Method!**